# 190 · LobS5 Interactive Explorer

Minimal notebook: loads only **LobS5** (aggressive_scenario) data from v3 grid.
Interactive Plotly master curves — hover to see config params & sample counts.

In [ ]:
import numpy as np
import pandas as pd
import re, math
from pathlib import Path
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
TICK_SIZE = 100
MAX_SAMPLES = 2048

# ── New-pipeline grid (Action 3) ──
# results/grid/<STOCK>-<MODEL>-<beta|relaxation>/<buy|sell>/exp_*/{data_cond,data_gen,aggressive_indices.csv}
def _find_lob_impact(start):
    for q in [start, *start.parents]:
        if (q / '3_scenarios' / 'results' / 'grid').exists():
            return q
    return start

LOB_IMPACT = _find_lob_impact(Path.cwd())
GRID_ROOT = LOB_IMPACT / '3_scenarios' / 'results' / 'grid'

# Daily H/L table → Parkinson sigma (newest run wins); keyed by (ticker, day).
_dhl = sorted((LOB_IMPACT / '2_daily_stats' / 'results').glob('daily_*/daily_h_l_all.csv'))
DHL_PATH = _dhl[-1] if _dhl else None
DAILY_HL = pd.read_csv(DHL_PATH) if DHL_PATH else pd.DataFrame()
_HL = {(str(r['ticker']), str(r['day'])): (float(r['highest_price']), float(r['lowest_price']))
       for _, r in DAILY_HL.iterrows()}

COLOR = '#D09A3C'
print(f'LOB_IMPACT : {LOB_IMPACT}')
print(f'GRID_ROOT  : {GRID_ROOT}  exists={GRID_ROOT.exists()}')
print(f'DAILY_HL   : {DHL_PATH}  ({len(_HL)} ticker-days)')


In [ ]:
# ── Data I/O helpers (new-pipeline grid layout) ──
LEAF_RE = re.compile(r'^([A-Za-z0-9]+)-([A-Za-z0-9]+)-(beta|relaxation)$')
FOLDER_PARAMS = {}   # folder name -> dict(stock, model, shape, i, c, mb, V)

def _latest_exp(side_dir):
    exps = sorted(p for p in side_dir.glob('exp_*') if p.is_dir())
    return exps[-1] if exps else None

def _read_cfg(exp_dir):
    import yaml
    f = exp_dir / 'config.yaml'
    return (yaml.safe_load(open(f)) or {}) if f.exists() else {}

def discover_grid(grid_root, stocks=None, models=None, shapes=None):
    rows = []
    for leaf in sorted(grid_root.iterdir()):
        if not leaf.is_dir() or leaf.name == '_configs':
            continue
        m = LEAF_RE.match(leaf.name)
        if not m:
            continue
        stock, model, shape = m.group(1), m.group(2), m.group(3)
        if (stocks and stock not in stocks) or (models and model not in models) \
           or (shapes and shape not in shapes):
            continue
        buy_exp, sell_exp = _latest_exp(leaf / 'buy'), _latest_exp(leaf / 'sell')
        if buy_exp is None or sell_exp is None:
            continue
        cfg = _read_cfg(buy_exp)
        # Map new identity onto the old (i, c, mb, V) schema so downstream cells just work.
        ins  = int(cfg.get('num_insertions', 0))
        cool = int(cfg.get('num_coolings', 0))
        mb   = int(cfg.get('n_gen_msgs', 0))
        vol  = int(cfg.get('order_volume', 0))
        rows.append({'folder': leaf.name, 'stock': stock, 'model': model, 'shape': shape,
                     'i': ins, 'c': cool, 'mb': mb, 'V': vol, 'Q_total': ins * vol,
                     'buy_path': buy_exp, 'sell_path': sell_exp})
        FOLDER_PARAMS[leaf.name] = {'stock': stock, 'model': model, 'shape': shape,
                                    'i': ins, 'c': cool, 'mb': mb, 'V': vol}
    return pd.DataFrame(rows)

def parse_folder_params(folder_name):
    p = FOLDER_PARAMS.get(folder_name)
    return (p['i'], p['c'], p['mb'], p['V']) if p else (None, None, None, None)

def compute_midprice(book_array):
    return (book_array[:, 0] + book_array[:, 2]) / 2

def load_aggressive_indices(data_path):
    f = data_path / 'aggressive_indices.csv'
    if not f.exists():
        return np.array([], dtype=int)
    return np.atleast_1d(np.loadtxt(f, dtype=int))

def discover_data_params(data_path, max_samples=None):
    cond_dir = data_path / 'data_cond'
    pat = re.compile(r'^(.+?)_(\d{4}-\d{2}-\d{2})_orderbook_real_id_(\d+)\.csv$')
    samples = []
    for f in cond_dir.glob('*_orderbook_real_id_*.csv'):
        m = pat.match(f.name)
        if m:
            samples.append((m.group(1), m.group(2), int(m.group(3))))
    samples.sort()
    if max_samples and len(samples) > max_samples:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(samples), size=max_samples, replace=False)
        samples = [samples[i] for i in sorted(idx)]
    return samples

def load_folder_data(data_path, max_samples=None):
    samples = discover_data_params(data_path, max_samples)
    gen_books, gen_msgs, cond_lens = {}, {}, {}
    for ticker, date, sid in samples:
        cond_bp = data_path / f'data_cond/{ticker}_{date}_orderbook_real_id_{sid}.csv'
        gen_bp  = data_path / f'data_gen/{ticker}_{date}_orderbook_real_id_{sid}_gen_id_0.csv'
        gen_mp  = data_path / f'data_gen/{ticker}_{date}_message_real_id_{sid}_gen_id_0.csv'
        if not gen_bp.exists():
            continue
        cond_book = np.loadtxt(cond_bp, delimiter=',')
        gen_book  = np.loadtxt(gen_bp, delimiter=',')
        gen_msg   = np.loadtxt(gen_mp, delimiter=',')
        cond_mp   = data_path / f'data_cond/{ticker}_{date}_message_real_id_{sid}.csv'
        cond_msg  = np.loadtxt(cond_mp, delimiter=',')
        key = (date, sid)
        cond_lens[key] = cond_book.shape[0]
        gen_books[key] = np.vstack([cond_book, gen_book])
        gen_msgs[key]  = np.vstack([cond_msg, gen_msg])
    return gen_books, gen_msgs, cond_lens

def load_all(grid_df):
    all_data = {}
    for _, row in tqdm(grid_df.iterrows(), total=len(grid_df), desc='Loading grid'):
        try:
            bb, bm, bc = load_folder_data(row['buy_path'],  MAX_SAMPLES)
            sb, sm, sc = load_folder_data(row['sell_path'], MAX_SAMPLES)
            all_data[row['folder']] = {
                'buy':  {'books': bb, 'msgs': bm, 'cond_lens': bc},
                'sell': {'books': sb, 'msgs': sm, 'cond_lens': sc},
            }
        except Exception as e:
            print(f'  ERR {row["folder"]}: {e}')
    return all_data

print('Helpers loaded.')


In [ ]:
# ── Discover the new grid & load ──
# Filters default to ALL 12 leaves. Mamba3 = neural model, Historic = replay baseline.
STOCKS = None    # e.g. ['EA', 'NVDA', 'AMD']
MODELS = None    # e.g. ['Mamba3', 'Historic']
SHAPES = None    # e.g. ['beta', 'relaxation']

grid = discover_grid(GRID_ROOT, STOCKS, MODELS, SHAPES).reset_index(drop=True)
print(f'Discovered {len(grid)} configs')
display(grid[['folder', 'stock', 'model', 'shape', 'i', 'c', 'mb', 'V', 'Q_total']])

data = load_all(grid)
print(f'Loaded {len(data)} folders')


In [ ]:
# ── Sigma-normalized master curves (volume time) ──

def compute_master_curve(buy_data, sell_data, folder, aggr_gen,
                         u_max=11.0, n_pts=500):
    i, c, mb, V = parse_folder_params(folder)
    stock = FOLDER_PARAMS.get(folder, {}).get('stock')
    if len(aggr_gen) < 2:
        return None
    s_gen = int(aggr_gen[0])
    e_gen = int(aggr_gen[-1])
    L = e_gen - s_gen
    if L == 0:
        return None
    bb, sb = buy_data['books'], sell_data['books']
    if not bb or not sb:
        return None
    min_len = min(min(b.shape[0] for b in bb.values()),
                  min(b.shape[0] for b in sb.values()))
    junction = list(buy_data['cond_lens'].values())[0]
    u_cap = min(u_max, (min_len - 1 - junction - s_gen) / L)
    if u_cap <= 0:
        return None
    u_grid = np.linspace(0, u_cap, n_pts)

    def side_impacts(books, conds):
        imps = []
        for sid, bk in books.items():
            date = sid[0]                       # date is embedded in the filename
            hl = _HL.get((stock, date))         # per-day H/L from daily_h_l_all.csv
            if hl is None:
                continue
            H = hl[0] / TICK_SIZE
            Lp = hl[1] / TICK_SIZE
            if H <= Lp or Lp <= 0:
                continue
            sigma = np.log(H / Lp) / 0.8325546   # Parkinson sigma
            if sigma <= 0:
                continue
            j = conds[sid]
            s_abs = j + s_gen
            if s_abs < 1 or s_abs >= min_len:
                continue
            mid = compute_midprice(bk[:min_len])
            ref = mid[s_abs - 1]
            if ref <= 0:
                continue
            raw = (mid[s_abs:min_len] - ref) / (ref * sigma)
            u_raw = np.arange(len(raw)) / L
            imps.append(np.interp(u_grid, u_raw, raw))
        return np.array(imps) if imps else None

    bi = side_impacts(bb, buy_data['cond_lens'])
    si = side_impacts(sb, sell_data['cond_lens'])
    if bi is None or si is None:
        return None
    mean = (np.mean(bi, axis=0) - np.mean(si, axis=0)) / 2
    std  = np.sqrt(np.std(bi, axis=0)**2 + np.std(si, axis=0)**2) / 2
    return {'u_grid': u_grid, 'combined_mean': mean, 'combined_std': std,
            'L': L, 'i': i, 'c': c, 'mb': mb, 'V': V, 'Q': i * V,
            'n_buy': bi.shape[0], 'n_sell': si.shape[0]}


# Compute all curves
curves = {}
for _, row in grid.iterrows():
    f = row['folder']
    if f not in data:
        continue
    aggr = load_aggressive_indices(row['buy_path'])
    cc = compute_master_curve(data[f]['buy'], data[f]['sell'], f, aggr)
    if cc is not None:
        curves[f] = cc

print(f'Computed {len(curves)} master curves')
for f, cc in sorted(curves.items(), key=lambda x: x[0]):
    print(f"  {f:28s}  i={cc['i']:3d} c={cc['c']:3d} mb={cc['mb']:4d} V={cc['V']:3d}  "
          f"L={cc['L']:4d}  n_buy={cc['n_buy']}  n_sell={cc['n_sell']}")


In [ ]:
# ── Interactive Master Curves (all configs, hover for details) ──

palette = px.colors.qualitative.D3 + px.colors.qualitative.Set2

# Adaptive u limit
u_all = [c['u_grid'][-1] for c in curves.values()]
U_PLOT = min(3.0, np.percentile(u_all, 10)) if u_all else 3.0

fig = go.Figure()

sorted_folders = sorted(curves.keys(),
    key=lambda f: (curves[f]['mb'], curves[f]['i'], curves[f]['V']))

for fi, folder in enumerate(sorted_folders):
    c = curves[folder]
    u, m, s = c['u_grid'], c['combined_mean'], c['combined_std']
    mask = u <= U_PLOT
    
    hover_text = (f"<b>{folder}</b><br>"
                  f"i={c['i']}, c={c['c']}, mb={c['mb']}, V={c['V']}<br>"
                  f"Q_total={c['Q']}, L={c['L']}<br>"
                  f"n_buy={c['n_buy']}, n_sell={c['n_sell']}<br>"
                  f"u=%{{x:.3f}}, I_norm=%{{y:.5f}}")
    
    fig.add_trace(go.Scatter(
        x=u[mask], y=m[mask], mode='lines',
        line=dict(color=palette[fi % len(palette)], width=2),
        name=f"i={c['i']} mb={c['mb']} V={c['V']}",
        hovertemplate=hover_text))

fig.add_vline(x=1.0, line_dash='dot', line_color='rgba(0,0,0,0.35)', line_width=1)

fig.update_layout(
    width=900, height=600,
    template='plotly_white',
    font=dict(family='Times New Roman, DejaVu Serif, serif', size=13, color='black'),
    title='LobS5 — Volume Time Master Curves (all configs)',
    margin=dict(l=60, r=15, t=40, b=55),
    legend=dict(font_size=10, bgcolor='rgba(255,255,255,0.9)',
                bordercolor='black', borderwidth=1),
    hovermode='closest',
)
fig.update_xaxes(title_text='u = n / L', showline=True, linewidth=1.5,
                 linecolor='black', mirror=True, ticks='outside')
fig.update_yaxes(title_text='I_norm(u)', showline=True, linewidth=1.5,
                 linecolor='black', mirror=True, ticks='outside')
fig.show()

In [ ]:
# ── Interactive Master Curves (all configs, hover for details) ──

palette = px.colors.qualitative.D3 + px.colors.qualitative.Set2

# Adaptive u limit
u_all = [c['u_grid'][-1] for c in curves.values()]
U_PLOT = min(3.0, np.percentile(u_all, 10)) if u_all else 3.0

fig = go.Figure()

sorted_folders = sorted(curves.keys(),
    key=lambda f: (curves[f]['mb'], curves[f]['i'], curves[f]['V']))

for fi, folder in enumerate(sorted_folders):
    c = curves[folder]
    u, m, s = c['u_grid'], c['combined_mean'], c['combined_std']
    mask = u <= U_PLOT
    
    hover_text = (f"<b>{folder}</b><br>"
                  f"i={c['i']}, c={c['c']}, mb={c['mb']}, V={c['V']}<br>"
                  f"Q_total={c['Q']}, L={c['L']}<br>"
                  f"n_buy={c['n_buy']}, n_sell={c['n_sell']}<br>"
                  f"u=%{{x:.3f}}, I_norm=%{{y:.5f}}")
    
    fig.add_trace(go.Scatter(
        x=u[mask], y=m[mask], mode='lines',
        line=dict(color=palette[fi % len(palette)], width=2),
        name=f"i={c['i']} mb={c['mb']} V={c['V']}",
        hovertemplate=hover_text))

fig.add_vline(x=1.0, line_dash='dot', line_color='rgba(0,0,0,0.35)', line_width=1)

fig.update_layout(
    width=900, height=600,
    template='plotly_white',
    font=dict(family='Times New Roman, DejaVu Serif, serif', size=13, color='black'),
    title='LobS5 — Volume Time Master Curves (all configs)',
    margin=dict(l=60, r=15, t=40, b=55),
    legend=dict(font_size=10, bgcolor='rgba(255,255,255,0.9)',
                bordercolor='black', borderwidth=1),
    hovermode='closest',
)
fig.update_xaxes(title_text='u = n / L', showline=True, linewidth=1.5,
                 linecolor='black', mirror=True, ticks='outside')
fig.update_yaxes(title_text='I_norm(u)', showline=True, linewidth=1.5,
                 linecolor='black', mirror=True, ticks='outside')
fig.show()

In [ ]:
# ── Average Master Curve with ±1σ band ──

u_common = np.linspace(0, U_PLOT, 300)
interps = []
labels = []
for f, c in curves.items():
    if c['u_grid'][-1] >= U_PLOT:
        interps.append(np.interp(u_common, c['u_grid'], c['combined_mean']))
        labels.append(f)

avg = np.mean(interps, axis=0)
std = np.std(interps, axis=0)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=np.concatenate([u_common, u_common[::-1]]),
    y=np.concatenate([avg + std, (avg - std)[::-1]]),
    fill='toself', fillcolor='rgba(208,154,60,0.15)',
    line=dict(width=0), showlegend=False, name='±1σ'))
fig2.add_trace(go.Scatter(
    x=u_common, y=avg, mode='lines',
    line=dict(color=COLOR, width=3), name=f'LobS5 avg ({len(interps)} curves)',
    hovertemplate='u=%{x:.3f}<br>I_norm=%{y:.5f}'))
fig2.add_vline(x=1.0, line_dash='dot', line_color='rgba(0,0,0,0.35)', line_width=1)

fig2.update_layout(
    width=700, height=450, template='plotly_white',
    font=dict(family='Times New Roman, DejaVu Serif, serif', size=13, color='black'),
    title='LobS5 — Average Master Curve',
    margin=dict(l=60, r=15, t=40, b=55),
    legend=dict(x=0.98, y=0.98, xanchor='right', yanchor='top',
                bgcolor='rgba(255,255,255,0.85)', bordercolor='black', borderwidth=1),
    hovermode='closest',
)
fig2.update_xaxes(title_text='u = n / L', showline=True, linewidth=1.5,
                  linecolor='black', mirror=True, ticks='outside')
fig2.update_yaxes(title_text='I_norm(u)', showline=True, linewidth=1.5,
                  linecolor='black', mirror=True, ticks='outside')
fig2.show()

In [ ]:
# ── Single trajectory explorer ──
# Pick a config with median insertions/cooling to start.
# Change FOLDER below to explore others.

# Show available configs sorted by (i, mb, V)
configs = grid[['folder','stock','model','shape','i','c','mb','V','Q_total']].sort_values(['stock','model','shape'])
display(configs)

# Pick the middle one by Q_total
mid_idx = len(configs) // 2
FOLDER = configs.iloc[mid_idx]['folder']
print(f'\nSelected: {FOLDER}')

In [ ]:
# ── Interactive: individual sample trajectories for selected config ──

d = data[FOLDER]
buy_books = d['buy']['books']
sell_books = d['sell']['books']
buy_conds = d['buy']['cond_lens']
sell_conds = d['sell']['cond_lens']

# Aggressive indices
row_sel = grid[grid['folder'] == FOLDER].iloc[0]
aggr_idx = load_aggressive_indices(row_sel['buy_path'])
i_val, c_val, mb_val, V_val = parse_folder_params(FOLDER)

print(f'Config: i={i_val}, c={c_val}, mb={mb_val}, V={V_val}')
print(f'Aggressive indices: {aggr_idx}')
print(f'Buy samples: {len(buy_books)}, Sell samples: {len(sell_books)}')

# Plot a subsample of individual midprice trajectories (buy side)
fig3 = go.Figure()
n_show = min(50, len(buy_books))  # show up to 50 trajectories
sids = list(buy_books.keys())[:n_show]

junction = list(buy_conds.values())[0]
min_len = min(b.shape[0] for b in buy_books.values())

for sid in sids:
    bk = buy_books[sid]
    mid = compute_midprice(bk[:min_len])
    # Normalize: show return from junction point
    ref = mid[junction]
    ret = (mid - ref) / ref * 1e4  # in bps
    
    fig3.add_trace(go.Scatter(
        x=np.arange(min_len), y=ret, mode='lines',
        line=dict(width=0.7, color='rgba(208,154,60,0.3)'),
        name=f'sample {sid}', showlegend=False,
        hovertemplate=f'sample={sid}<br>step=%{{x}}<br>return=%{{y:.2f}} bps'))

# Mark aggressive insertion points
for ai in aggr_idx:
    abs_idx = junction + ai
    if abs_idx < min_len:
        fig3.add_vline(x=abs_idx, line_dash='dot',
                       line_color='rgba(255,0,0,0.3)', line_width=0.8)

fig3.add_vline(x=junction, line_dash='dash', line_color='blue', line_width=1.5)
fig3.add_annotation(x=junction, y=0, text='junction', showarrow=False,
                    yshift=15, font_color='blue', font_size=10)

fig3.update_layout(
    width=900, height=500, template='plotly_white',
    font=dict(family='Times New Roman, DejaVu Serif, serif', size=13),
    title=f'LobS5 — Individual trajectories (buy) — {FOLDER}',
    margin=dict(l=60, r=15, t=40, b=55),
    hovermode='closest',
)
fig3.update_xaxes(title_text='Message step', showline=True, linewidth=1.5,
                  linecolor='black', mirror=True, ticks='outside')
fig3.update_yaxes(title_text='Midprice return (bps from junction)', showline=True,
                  linewidth=1.5, linecolor='black', mirror=True, ticks='outside')
fig3.show()

---
## Как пользоваться

1. **Master Curves** (ячейка `interactive_master`) — наведи мышкой на кривую, чтобы увидеть параметры конфига (i, c, mb, V), число buy/sell samples, Q_total
2. **Single trajectory** — измени `FOLDER` в ячейке `single_trajectory` на нужный конфиг и перезапусти 2 ячейки ниже
3. Красные пунктирные линии = точки вставки aggressive orders, синяя = junction (конец conditioning, начало генерации)

In [ ]:
# ── Midprice plot for paper — single axis, skip 0–450 ──
import matplotlib.pyplot as plt

# FOLDER_KEY = 'i5_c50_mb5_v75_cntxt55%'

#4 , 20

FOLDER_KEY = configs['folder'].iloc[len(configs) // 2]

d = data[FOLDER_KEY]
row_k = grid[grid['folder'] == FOLDER_KEY].iloc[0]
aggr = load_aggressive_indices(row_k['buy_path'])
i_val, c_val, mb_val, V_val = parse_folder_params(FOLDER_KEY)

# Average midprice across all buy samples
buy_books = d['buy']['books']
buy_conds = d['buy']['cond_lens']
junction = list(buy_conds.values())[0]  # =500
min_len = min(b.shape[0] for b in buy_books.values())

mids = []
for sid, bk in buy_books.items():
    mid = compute_midprice(bk[:min_len])
    ref = mid[junction]
    if ref > 0:
        mids.append((mid - ref) / ref * 1e4)
avg_mid = np.mean(mids, axis=0)

# Rolling average window (number of messages)
SMOOTH_W = 2
avg_smooth = np.convolve(avg_mid, np.ones(SMOOTH_W) / SMOOTH_W, mode='same')

# Visible range: skip 0–X_START, show X_START–N_SHOW
X_START = 490
N_GEN = 12500
N_SHOW = min(junction + N_GEN, min_len)

x = np.arange(X_START, N_SHOW)
y = avg_smooth[X_START:N_SHOW]

# ── Plot ──
fig, ax = plt.subplots(figsize=(5.5, 3.8), facecolor='white')

# Context shaded region (X_START to junction)
ax.axvspan(X_START, junction, color='#5B8DB8', alpha=0.06, zorder=0)

# Junction line
ax.axvline(x=junction, color='#5B8DB8', linestyle='--', linewidth=1.0, alpha=0.6)

# Midprice curve (black)
ax.plot(x, y, color='black', linewidth=1.5, solid_capstyle='round')

# Insertion markers (blue dots on the curve)
for ai in aggr:
    abs_i = junction + ai - 1
    if X_START <= abs_i < N_SHOW:
        ax.plot(abs_i, avg_smooth[abs_i], 'o', color='#2E6EB5',
                markersize=6, markeredgewidth=0, zorder=5)

# Style
ax.set_xlabel('message index  $n$', fontsize=11, fontfamily='serif')
ax.set_ylabel('$\\Delta\\,\\mathrm{mid}$  (bps)', fontsize=11, fontfamily='serif')
ax.tick_params(labelsize=9.5, direction='out')
for spine in ax.spines.values():
    spine.set_linewidth(1.0)
    spine.set_color('black')
ax.set_xlim(X_START, N_SHOW)

# Break marks on left edge (indicate skipped 0–X_START)
d_brk = 0.012
bkw = dict(transform=ax.transAxes, color='black', clip_on=False, linewidth=0.8)
ax.plot((-d_brk, +d_brk), (-d_brk, +d_brk), **bkw)
ax.plot((-d_brk, +d_brk), (1 - d_brk, 1 + d_brk), **bkw)

fig.tight_layout(pad=0.8)
# fig.savefig('pics_for_190/midprice_small.png',
#             dpi=300, bbox_inches='tight', facecolor='white')
print(f'SMOOTH_W={SMOOTH_W}, aggressive_indices: {aggr.tolist()}')
print(f'dot positions (abs): {[junction + ai - 1 for ai in aggr]}')
plt.show()

In [ ]:
# ── Synthetic schematic: market impact curve (publication style) ──
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def synthetic_impact_curve(n_ctx=80, n_ins=5, mb=10, n_cool=160,
                           jump=1.0, decay_frac=0.25, final_frac=0.5,
                           seed=42):
    rng = np.random.RandomState(seed)

    # ── Context: flat at base price ──
    ctx = np.zeros(n_ctx)

    # ── Insertion phase ──
    ins = []
    level = 0.0
    dot_xs, dot_ys = [], []

    for k in range(n_ins):
        # Jump (concave / diminishing)
        step = jump * (0.75 ** k)
        level += step
        # Dot BEFORE the peak of this jump (on the rising edge)
        dot_xs.append(n_ctx + len(ins))
        dot_ys.append(level - step * 0.15)   # slightly below the local peak
        ins.append(level)

        # Partial relaxation between insertions
        target = level - step * decay_frac
        t = np.linspace(0, 1, mb)
        seg = level + (target - level) * (1 - np.exp(-3.5 * t))
        ins.extend(seg.tolist())
        level = seg[-1]

    peak = max(ins)
    peak_x = n_ctx + ins.index(peak)

    # ── Cooling: power-law-ish decay to final_frac * peak ──
    final_level = peak * final_frac
    t = np.linspace(0, 1, n_cool)
    cool = level + (final_level - level) * (1 - np.exp(-3.0 * t))

    # ── Permanent impact: smooth concave rise → flat ──
    total = len(ctx) + len(ins) + len(cool)
    perm = np.zeros(total)
    rise_len = len(ctx) + len(ins)
    t_rise = np.linspace(0, 1, rise_len - n_ctx)
    perm_curve = final_level * (1 - np.exp(-2.5 * t_rise))
    perm[n_ctx:rise_len] = perm_curve
    perm[rise_len:] = final_level

    # ── Assemble total impact ──
    y = np.concatenate([ctx, np.array(ins), cool])
    # Tiny noise + smooth
    y += rng.normal(0, 0.008, len(y))
    y = np.convolve(y, np.ones(3) / 3, mode='same')
    y[:n_ctx] = 0  # keep context perfectly flat
    perm = np.convolve(perm, np.ones(5) / 5, mode='same')

    x = np.arange(len(y))
    return x, y, perm, np.array(dot_xs), np.array(dot_ys), n_ctx, peak_x, peak, final_level


# ── Build data ──
x, y, perm, dx, dy, n_ctx, peak_x, peak_val, final_val = synthetic_impact_curve()

# ── Plot ──
fig, ax = plt.subplots(figsize=(7, 4.2), facecolor='white')

# Context region
ax.axvspan(0, n_ctx, color='#E8E8E8', alpha=0.5, zorder=0)

# Permanent impact (dotted)
ax.plot(x, perm, color='black', linewidth=2.5, linestyle=(0, (2, 2.5)),
        zorder=3, label='Permanent Impact')

# Total / temporary impact (solid)
ax.plot(x, y, color='black', linewidth=2.8, solid_capstyle='round',
        zorder=4, label='Temporary Impact')

# Child order dots (skip the last one so it's not on the peak)
ax.scatter(dx[:-1], dy[:-1], s=70, color='#7CB342', zorder=6,
           edgecolors='none', label='Child Orders')

# ── Annotations ──
# t1, t2 labels
ax.annotate('$t_1$', xy=(n_ctx, 0), xytext=(n_ctx, -0.22),
            fontsize=13, fontfamily='serif', ha='center', va='top')
ax.annotate('$t_2$', xy=(peak_x, 0), xytext=(peak_x, -0.22),
            fontsize=13, fontfamily='serif', ha='center', va='top')

# Vertical dashed line at t2 (end of execution)
ax.axvline(x=peak_x, color='black', linestyle='--', linewidth=1.2, zorder=2)

# "Metaorder" arrow
arr_y = peak_val * 1.08
ax.annotate('', xy=(peak_x - 1, arr_y), xytext=(n_ctx + 1, arr_y),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
ax.text((n_ctx + peak_x) / 2, arr_y + 0.08, 'Metaorder',
        ha='center', va='bottom', fontsize=12, fontfamily='serif',
        fontweight='bold')

# "Context" label
ax.text(n_ctx / 2, peak_val * 0.55, 'Context',
        ha='center', va='center', fontsize=12, fontfamily='serif',
        color='#555555', fontstyle='italic')

# ── Legend ──
handles = [
    mpatches.Patch(facecolor='#7CB342', edgecolor='none', label='Child Orders'),
    plt.Line2D([], [], color='black', linewidth=2.5, label='Temporary Impact'),
    plt.Line2D([], [], color='black', linewidth=2.5, linestyle=(0, (2, 2.5)),
               label='Permanent Impact'),
]
ax.legend(handles=handles, loc='upper right', fontsize=10, frameon=True,
          fancybox=False, edgecolor='black', framealpha=1,
          prop={'family': 'serif'})

# ── Style ──
ax.set_xlabel('Time', fontsize=13, fontfamily='serif', fontweight='bold')
ax.set_ylabel('Price', fontsize=13, fontfamily='serif', fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlim(-5, len(x) + 5)
ax.set_ylim(-0.35, peak_val * 1.32)

# Thick axis spines with arrows
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
for spine in ['bottom', 'left']:
    ax.spines[spine].set_linewidth(2.0)
    ax.spines[spine].set_color('black')

# Arrow tips on axes
ax.plot(1, 0, '^k', transform=ax.get_yaxis_transform(), markersize=8,
        clip_on=False, zorder=10)
ax.plot(len(x) + 5, 0, '>k', transform=ax.get_xaxis_transform(),
        markersize=8, clip_on=False, zorder=10)

fig.tight_layout(pad=0.8)
_OUT = Path('results/explorer_figs'); _OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(_OUT / 'impact_schematic.png',
            dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(_OUT / 'impact_schematic.pdf',
            bbox_inches='tight', facecolor='white')
print(f'Saved → {_OUT}/impact_schematic.{{png,pdf}}')
plt.show()